# 실습 1 — BERT 풀 파인튜닝 (감성 분석)

사전학습된 **DistilBERT** 모델을 IMDB 영화 리뷰 데이터로 파인튜닝하여 감성 분류 모델을 만듭니다.

| 항목 | 내용 |
|---|---|
| **모델** | `distilbert-base-uncased` (66M 파라미터) |
| **데이터셋** | IMDB 영화 리뷰 (50개 샘플) |
| **태스크** | 감성 분류 (긍정 / 부정) |
| **방법** | Full Fine-tuning (전체 파라미터 학습) |

> **인코더 모델 vs 디코더 모델**  
> DistilBERT는 인코더 전용 모델로, 텍스트를 생성하는 대신 입력 문장 전체를 한 번에 이해하는 데 특화되어 있습니다.  
> 분류, 감성 분석 같은 태스크에 적합합니다.

## ⚙️ Colab GPU 설정

실습 전 반드시 GPU를 활성화해야 합니다.

**설정 방법:**
1. 상단 메뉴 → **런타임** 클릭
2. **런타임 유형 변경** 클릭
3. 하드웨어 가속기 → **T4 GPU** 선택
4. **저장** 후 런타임 재시작

> ⚠️ GPU 없이 실행하면 학습 속도가 매우 느립니다. 반드시 설정 후 시작하세요.

In [1]:
# GPU 활성화 확인 — 반드시 GPU가 표시되어야 합니다
import torch

print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 모델  : {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU가 활성화되지 않았습니다. 위의 설정 방법을 따라주세요.")

GPU 사용 가능: True
GPU 모델  : Tesla T4
GPU 메모리: 15.6 GB


## 📦 라이브러리 설치

| 라이브러리 | 역할 |
|---|---|
| `transformers` | 사전학습 모델 로딩 및 파인튜닝 도구 |
| `datasets` | HuggingFace 데이터셋 로딩 |
| `scikit-learn` | 정확도(accuracy) 계산 |

In [2]:
!pip install -q transformers datasets scikit-learn
print("✅ 설치 완료")

✅ 설치 완료


## 1단계. 데이터 준비

**IMDB 데이터셋**은 영화 리뷰와 감성 레이블(긍정/부정)로 구성된 NLP 대표 데이터셋입니다.

- 전체 크기: 학습 25,000개 / 테스트 25,000개
- 이번 실습: 빠른 실습을 위해 **50개만** 사용 (학습 40개 / 테스트 10개)

> **왜 데이터를 나눌까?**  
> 학습 데이터로 모델을 훈련하고, 테스트 데이터로 모델이 새로운 입력을 얼마나 잘 처리하는지 검증합니다.  
> 학습에 사용한 데이터로 테스트하면 모델이 답을 외운 것인지 실제로 이해한 것인지 구별할 수 없습니다.

In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb", split="train")

# 먼저 전체 train을 섞고 나서 500개 선택
dataset = dataset.shuffle(seed=42).select(range(500))

# 그 다음 train/test split
dataset = dataset.train_test_split(test_size=0.2, seed=42)

from collections import Counter
print("train label 분포:", Counter(dataset["train"]["label"]))
print("test label 분포 :", Counter(dataset["test"]["label"]))

print(f"학습 데이터 수: {len(dataset['train'])}")
print(f"테스트 데이터 수: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train label 분포: Counter({0: 204, 1: 196})
test label 분포 : Counter({0: 50, 1: 50})
학습 데이터 수: 400
테스트 데이터 수: 100


In [4]:
# 샘플 하나 확인
sample = dataset["train"][0]
print("[리뷰 내용 앞 300자]")
print(sample['text'][:300])
print(f"\n레이블: {sample['label']} → {'긍정(Positive)' if sample['label'] == 1 else '부정(Negative)'}")

[리뷰 내용 앞 300자]
All right I recently got a chance to rent this and watch Santa Claus conquers the martains. Although the children were much more sadistic in SCCTM, I would have to say that Santa Claus was a much worse movie. As a spanish assignment in Spanish 5 we all had to watch it. I'll tell you, usually when we

레이블: 0 → 부정(Negative)


## 2단계. 토크나이저 및 모델 준비

**토크나이저(Tokenizer)**는 텍스트를 모델이 이해할 수 있는 숫자(토큰 ID)로 변환합니다.

```
"I love this movie" → [101, 1045, 2293, 2023, 3185, 102]
                           ↑                              ↑
                       [CLS] 시작 토큰            [SEP] 끝 토큰
```

> 반드시 **모델과 같은 토크나이저**를 사용해야 합니다.  
> 토크나이저가 다르면 같은 단어도 다른 숫자로 변환되어 모델이 제대로 동작하지 않습니다.

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# num_labels=2: 분류할 클래스 수 (부정=0, 긍정=1)

print(f"✅ 모델 로딩 완료: {MODEL_NAME}")
print(f"전체 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ 모델 로딩 완료: distilbert-base-uncased
전체 파라미터 수: 66,955,010


## 3단계. 데이터 전처리 (텍스트 → 토큰)

---
### ✏️ 빈칸 1, 2 — tokenize 함수 완성

`tokenizer()` 함수의 두 옵션을 채워보세요.

| 옵션 | 역할 | 채울 값 |
|---|---|---|
| `truncation` | 512 토큰을 초과하는 문장을 잘라냄 | `True` 또는 `False` |
| `padding` | 짧은 문장을 0으로 채워 길이를 통일 | `True` 또는 `False` |

**힌트:** 두 옵션 모두 활성화해야 배치(여러 문장을 한 번에) 처리가 가능합니다.

---

In [6]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,  # ✏️ 빈칸 1
        padding=True,     # ✏️ 빈칸 2
    )

dataset = dataset.map(tokenize, batched=True)

print("✅ 전처리 완료")
print(f"input_ids 예시: {dataset['train'][0]['input_ids'][:10]} ...")
print(f"토큰 총 개수  : {len(dataset['train'][0]['input_ids'])}개")

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

✅ 전처리 완료
input_ids 예시: [101, 2035, 2157, 1045, 3728, 2288, 1037, 3382, 2000, 9278] ...
토큰 총 개수  : 512개


## 4단계. 훈련 설정

---
### ✏️ 빈칸 3, 4, 5 — 하이퍼파라미터와 metrics 함수 완성

아래 표를 참고해서 빈칸을 채워보세요.

| 파라미터 | 권장값 | 설명 |
|---|---|---|
| `per_device_train_batch_size` | `8` | GPU 하나가 한 번에 처리할 샘플 수 |
| `num_train_epochs` | `15` | 전체 데이터를 반복 학습하는 횟수 |
| `argmax` 메서드 | `argmax` | logits에서 가장 높은 점수의 인덱스 추출 |

---

In [7]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # ✏️ 빈칸 3: logits에서 가장 높은 점수의 인덱스를 예측값으로 사용
    # logits shape: (batch_size, 2) → [부정 점수, 긍정 점수]
    predictions = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

args = TrainingArguments(
    output_dir="./bert-finetuned",
    per_device_train_batch_size=8,  # ✏️ 빈칸 4: 배치 크기
    num_train_epochs=5,             # ✏️ 빈칸 5: 에폭 수
    report_to="none",
    logging_steps=1,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

print("✅ Trainer 구성 완료")

✅ Trainer 구성 완료


## 5단계. 파인튜닝 실행

`trainer.train()` 한 줄로 아래 과정이 에폭마다 자동 반복됩니다.

```
입력 데이터 → 순전파(Forward) → 손실(Loss) 계산 → 역전파(Backward) → 파라미터 업데이트
```

학습 중 **`loss` 값이 점점 줄어드는** 것을 확인해보세요.

In [8]:
trainer.train()

Step,Training Loss
1,0.691433
2,0.784582
3,0.652060
4,0.679528
5,0.799766
6,0.671464
7,0.729355
8,0.675176
9,0.733404
10,0.743728


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.20862177366018295, metrics={'train_runtime': 109.493, 'train_samples_per_second': 18.266, 'train_steps_per_second': 2.283, 'total_flos': 264934797312000.0, 'train_loss': 0.20862177366018295, 'epoch': 5.0})

### 📊 학습 결과 해석

- **`loss`** 값이 낮아질수록 모델이 훈련 데이터를 잘 학습하고 있다는 의미입니다.
- 50개의 작은 데이터셋이라 loss가 0에 가까워질 수 있는데, 이는 **과적합(Overfitting)** 신호일 수 있습니다.
  - 과적합: 훈련 데이터는 잘 맞히지만 새로운 데이터는 못 맞히는 현상
- 실제 서비스에서는 수천~수만 개의 데이터로 학습합니다.

## 6단계. 새 문장 예측

---
### ✏️ 빈칸 6, 7, 8 — 예측 코드 완성

---

In [9]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ✏️ 빈칸 6: 모델을 평가(추론) 모드로 전환
# 힌트: 학습 모드(train)와 반대 모드의 이름은?
# → 평가 모드에서는 Dropout 같은 랜덤 요소가 꺼져 예측이 일관됩니다
model.eval()

text = "I would put this at the top of my list of films in the category of unwatchable trash!"
inputs = tokenizer(text, return_tensors="pt").to(device)

# torch.no_grad(): 추론 시 gradient 계산을 끄면 메모리가 절약됩니다
with torch.no_grad():
    output = model(**inputs)
    label = output.logits.argmax(-1).item()

# ✏️ 빈칸 : 긍정 레이블의 번호 (0=부정, 1=긍정)
print(f"예측 결과: {'긍정 ✅' if label == 1 else '부정 ❌'}")
print(f"리뷰: {text[:60]}...")

예측 결과: 부정 ❌
리뷰: I would put this at the top of my list of films in the categ...


In [10]:
# 여러 문장으로 직접 테스트해보기
test_texts = [
    "This movie was absolutely wonderful! A masterpiece.",
    "What a waste of time. Terrible acting and a boring plot.",
    "It was okay, nothing special but not bad either.",
]

print("=" * 60)
for text in test_texts:
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model(**inputs)
        label = output.logits.argmax(-1).item()
    result = "긍정 ✅" if label == 1 else "부정 ❌"
    print(f"{result} | {text[:55]}...")
print("=" * 60)

긍정 ✅ | This movie was absolutely wonderful! A masterpiece....
부정 ❌ | What a waste of time. Terrible acting and a boring plot...
부정 ❌ | It was okay, nothing special but not bad either....


## ✅ 실습 1 완료!

### 전체 흐름 정리

```
데이터 로딩 → 토크나이징 → 모델 로딩 → Trainer 구성 → 학습 → 예측
```

| 단계 | 핵심 코드 | 역할 |
|---|---|---|
| 전처리 | `tokenizer(text, truncation=True, padding=True)` | 텍스트 → 토큰 변환 |
| 학습 | `trainer.train()` | 파인튜닝 실행 |
| 예측 | `logits.argmax(-1)` | 클래스 예측 |

### 다음 실습
실습 2에서는 더 큰 생성형 모델(Qwen2.5-3B)을 **LoRA**로 효율적으로 파인튜닝합니다.  
전체 파라미터의 단 **1%만 학습**해도 좋은 성능을 낼 수 있습니다!